In [9]:
from langchain.agents import initialize_agent, Tool, AgentType
from langchain.agents import AgentExecutor
from langchain_ollama.llms import OllamaLLM
from langchain_ollama.chat_models import ChatOllama
# from langchain_community.embeddings import OllamaEmbeddings
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
from langchain_neo4j import Neo4jGraph
from neo4j import GraphDatabase

# Initialize the Neo4j database connection
uri = "bolt://localhost:7687"
user = "neo4j"
password = "password"

driver = GraphDatabase.driver(uri, auth=(user, password))

# Define a custom Neo4j query tool
def query_neo4j_tool(query: str) -> str:
    session = driver.session()
    result = session.run(query)
    result_data = [record for record in result]
    session.close()
    return str(result_data)

# Define the LLM (using OpenAI in this case)
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    # other params...
)

# embeddings = OllamaEmbeddings(
#     model="llama3.2",
# )

# Define the memory (to hold the conversation context)
memory = ConversationBufferMemory(memory_key="chat_history")

# Define the Neo4j tool for LangChain
neo4j_tool = Tool(
    name="Neo4j Query Tool",
    func=query_neo4j_tool,
    description="Use this tool to query the Neo4j Knowledge Graph for information."
)

# Define the prompt for the agent
prompt_template = """You are an intelligent agent tasked with answering questions using the Neo4j Knowledge Graph.
You will first decompose the question into relevant queries and then use the Neo4j tool to fetch the necessary information.
Provide the final answer based on the retrieved data.

User Question: {input}
"""

# Set up the LLM chain and agent with memory and Neo4j tool
llm_chain = LLMChain(prompt=PromptTemplate(template=prompt_template, input_variables=["input"]), llm=llm)

tools = [neo4j_tool]

agent = initialize_agent(
    tools, 
    llm, 
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, 
    memory=memory
)

# Example user query
query = "Who likes to eat sweet things during tea?"

# Run the agent to handle the query and return an answer
answer = agent.run(query)
print(answer)

## The schema would help; prompt needs improvement




> Entering new AgentExecutor chain...
Thought: To find out who likes to eat sweet things during tea, we need to query the Neo4j Knowledge Graph for people who have a preference for sweets and also enjoy having tea.

Action: Use the Neo4j Query Tool
Action Input: "MATCH (p:Person)-[:LIKES]->(s:Sweets) WHERE s.name IN ['sweet', 'dessert'] AND p.hasPreference('tea') RETURN p"
Observation: Use the Neo4j Query Tool is not a valid tool, try one of [Neo4j Query Tool].
Thought:Question: Who likes to eat sweet things during tea?
Thought: Thought: To find out who likes to eat sweet things during tea, we need to query the Neo4j Knowledge Graph for people who have a preference for sweets and also enjoy having tea.

Action: Use the Neo4j Query Tool
Action Input: "MATCH (p:Person)-[:LIKES]->(s:Sweets) WHERE s.name IN ['sweet', 'dessert'] AND p.hasPreference('tea') RETURN p"
Observation: Use the Neo4j Query Tool is not a valid tool, try one of [Neo4j Query Tool].
Thought:Question: Who likes to eat 

KeyboardInterrupt: 